# TP1/TP1 Concurrent Analysis

## Multiple Linear Regression for Canonical Proportion as a Predictor of Standardized Speech Assessments

The following notebook contains code to perform multiple linear regression for canonical proportion (CP) as a predictor of the following standardized speech assessments:

- GFTA
- EVT

The following variables were used as covariates:

- Gender (dummy)
- Age (in months)
- Maternal education (on 1-7 scale - see paper for more detail.)

## GFTA

Adding scaled cp and outcome variables to 

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# === Load your file ===
df = pd.read_csv("tp1_tp1_master.csv")  # adjust path if needed

# --- Ensure numeric ---
for c in ["GFTA_Standard", "canonical_proportion"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Drop rows missing values for scaling ---
mask = df[["GFTA_Standard", "canonical_proportion"]].notna().all(axis=1)
df_scale = df.loc[mask].copy()

# --- Scale GFTA and CP ---
scaler = StandardScaler()
df_scale[["GFTA_Standard_scaled", "canonical_proportion_scaled"]] = scaler.fit_transform(
    df_scale[["GFTA_Standard", "canonical_proportion"]]
)

# --- Merge back (NaN for rows that had missing values) ---
df = df.merge(df_scale[["GFTA_Standard_scaled", "canonical_proportion_scaled"]],
              left_index=True, right_index=True, how="left")

# --- Save to new CSV ---
df.to_csv("tp1_tp1_master_with_scaled.csv", index=False)

print("File saved as tp1_tp1_master_with_scaled.csv")


### Unweighted, Unscaled - [Expanded, Unscaled]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2

# === Load data ===
df = pd.read_csv("tp1_tp1_master.csv")  # adjust path to TP1 dataset

# --- Coerce numeric columns ---
num_cols = ["GFTA_Standard", "canonical_proportion", "age_mos", "Maternal_Education_Level"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    # numeric 0/1
    try:
        v = float(x)
        if v in (0.0, 1.0):
            return int(v)  # 0=Male, 1=Female
    except Exception:
        pass
    # strings
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields ---
needed = ["GFTA_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy"]
df_clean = df.dropna(subset=needed).copy()

# --- Center & scale CP only (z-score over the analysis sample) ---
cp_mean = df_clean["canonical_proportion"].mean()
cp_std  = df_clean["canonical_proportion"].std(ddof=0)  # population SD; use ddof=1 if preferred
df_clean["canonical_proportion_scaled"] = (df_clean["canonical_proportion"] - cp_mean) / cp_std

# --- Predictors (age, gender, maternal ed unscaled; CP scaled) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

# Ensure valid rows for both models
valid_rows = df_clean[
    ["GFTA_Standard"] + predictors_exp
].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_rows].copy()

def fit_model(outcome_var, predictors, data):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    return sm.OLS(y.loc[mask], X.loc[mask]).fit()

# --- Fit models ---
gfta_baseline = fit_model("GFTA_Standard", predictors_base, df_clean)
gfta_expanded = fit_model("GFTA_Standard", predictors_exp, df_clean)

# --- Likelihood ratio test ---
ll_base = gfta_baseline.llf
ll_exp  = gfta_expanded.llf
k_base  = len(gfta_baseline.params)
k_exp   = len(gfta_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model (UNWEIGHTED, UNSCALED predictors): Demographics → GFTA_Standard")
print(gfta_baseline.summary(), "\n")

print("Expanded Model (UNWEIGHTED): Demographics + CP(z) → GFTA_Standard")
print(gfta_expanded.summary(), "\n")

# --- CP effect report (on z-scored CP) ---
term = "canonical_proportion_scaled"
if term in gfta_expanded.params.index:
    cp_beta = gfta_expanded.params[term]
    cp_se   = gfta_expanded.bse[term]
    cp_t    = gfta_expanded.tvalues[term]
    cp_p    = gfta_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = gfta_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion (z-scored) effect in Expanded Model (predicting GFTA_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of GFTA (p < .05)."
          if cp_p < 0.05 else
          "❌ CP is not a statistically significant predictor of GFTA (p ≥ .05).")
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_p < 0.05 else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Sample size ---
print(f"\nN used in both models: {df_clean.shape[0]}")


### Weighted, Scaled - [Expanded, Scaled]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2

# === Load CSV ===
df = pd.read_csv("tp1_tp1_master.csv")  # <-- replace with actual file path

# --- Compute num_clips by summing columns 0 and 1 ---
df["num_clips"] = df[["0", "1"]].sum(axis=1)

# --- Coerce numeric columns ---
num_cols = [
    "GFTA_Standard", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields BEFORE scaling ---
needed = ["GFTA_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.dropna(subset=needed).copy()

# --- Prepare weights from num_clips (normalize to mean=1) ---
w = df_clean["num_clips"].astype(float).replace([np.inf, -np.inf], np.nan)
w = w.where(w > 0)
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()
df_clean["__w__"] = w

# --- Scale predictors ---
scaler_age = StandardScaler()
scaler_med = StandardScaler()
scaler_cp  = StandardScaler()

df_clean["age_mos_scaled"] = scaler_age.fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = scaler_med.fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = scaler_cp.fit_transform(
    df_clean[["canonical_proportion"]]
)

# --- Final sanity: drop any non-finite values and align weights ---
predictors_base = ["age_mos_scaled", "gender_dummy", "Maternal_Education_Level_scaled"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

cols_needed_all = ["GFTA_Standard"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS weighted by num_clips) ---
gfta_baseline = fit_model("GFTA_Standard", predictors_base, df_clean, weights)
gfta_expanded = fit_model("GFTA_Standard", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = gfta_baseline.llf
ll_exp  = gfta_expanded.llf
k_base  = len(gfta_baseline.params)
k_exp   = len(gfta_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips → mean-normalized]: Demographics → GFTA_Standard")
print(gfta_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips → mean-normalized]: Demographics + CP → GFTA_Standard")
print(gfta_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion_scaled"
if term in gfta_expanded.params.index:
    cp_beta = gfta_expanded.params[term]
    cp_se   = gfta_expanded.bse[term]
    cp_t    = gfta_expanded.tvalues[term]
    cp_p    = gfta_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = gfta_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting GFTA_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of GFTA (p < .05).\n"
          if cp_p < 0.05 else
          "❌ CP is not a statistically significant predictor of GFTA (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_p < 0.05 else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


In [ ]:
# --- Scale outcome (GFTA_Standard) as well ---
from sklearn.preprocessing import StandardScaler

scaler_gfta = StandardScaler()
df_clean["GFTA_Standard_scaled"] = scaler_gfta.fit_transform(
    df_clean[["GFTA_Standard"]]
)

# --- Update model fitting to use scaled outcome ---
gfta_baseline = fit_model("GFTA_Standard_scaled", predictors_base, df_clean, weights)
gfta_expanded = fit_model("GFTA_Standard_scaled", predictors_exp,  df_clean, weights)

# --- CP effect report (now on standardized GFTA) ---
term = "canonical_proportion_scaled"
if term in gfta_expanded.params.index:
    cp_beta = gfta_expanded.params[term]
    cp_se   = gfta_expanded.bse[term]
    cp_t    = gfta_expanded.tvalues[term]
    cp_p    = gfta_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = gfta_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting GFTA_Standard *scaled*):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2

# === Load updated file with scaled vars ===
df = pd.read_csv("tp1_tp1_master_with_scaled.csv")

# --- Compute num_clips (0 + 1) ---
df["num_clips"] = df[["0","1"]].sum(axis=1)

# --- Robust gender dummy ---
def to_gender_dummy(x):
    try:
        v = float(x)
        if v in (0.0, 1.0):
            return int(v)
    except Exception:
        pass
    s = str(x).strip().lower()
    if s in ("f","female"): return 1
    if s in ("m","male"):   return 0
    return np.nan
df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Keep only valid rows ---
needed = ["GFTA_Standard_scaled","age_mos","gender_dummy",
          "Maternal_Education_Level","canonical_proportion_scaled","num_clips"]
df_clean = df.dropna(subset=needed).copy()

# --- Weights: mean-normalized num_clips ---
w = df_clean["num_clips"].astype(float).clip(lower=1)
w = w / w.mean()
df_clean["__w__"] = w
weights = df_clean["__w__"].values

# --- Define predictors ---
predictors_base = ["age_mos","gender_dummy","Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models ---
gfta_baseline = fit_model("GFTA_Standard_scaled", predictors_base, df_clean, weights)
gfta_expanded = fit_model("GFTA_Standard_scaled", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = gfta_baseline.llf
ll_exp  = gfta_expanded.llf
k_base  = len(gfta_baseline.params)
k_exp   = len(gfta_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output ---
print("Baseline Model [WLS weighted by num_clips]: Demographics → GFTA (scaled)")
print(gfta_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips]: Demographics + CP → GFTA (scaled)")
print(gfta_expanded.summary(), "\n")

term = "canonical_proportion_scaled"
if term in gfta_expanded.params.index:
    cp_beta = gfta_expanded.params[term]
    cp_se   = gfta_expanded.bse[term]
    cp_t    = gfta_expanded.tvalues[term]
    cp_p    = gfta_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = gfta_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting GFTA scaled):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")
    print("CP significance:",
          "✅ Significant (p < .05)\n" if cp_p < 0.05 else
          "❌ Not significant (p ≥ .05)\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"LRT Statistic: {lrt_stat:.4f}, df={lrt_df}, p={lrt_p:.6f}")
print("Model comparison:",
      "✅ Expanded model significantly improves fit"
      if lrt_p < 0.05 else
      "❌ Expanded model does not significantly improve fit")

print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary: mean={weights.mean():.3f}, min={weights.min():.3f}, max={weights.max():.3f}")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2

# === Load CSV ===
df = pd.read_csv("tp1_tp1_master.csv")  # <-- replace with actual file path

# --- Compute num_clips by summing columns 0 and 1 ---
df["num_clips"] = df[["0", "1"]].sum(axis=1)

# --- Coerce numeric columns ---
num_cols = [
    "GFTA_Standard", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    try:
        v = float(x)
        if v in (0.0, 1.0):
            return int(v)  # already numeric 0/1
    except Exception:
        pass
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields BEFORE scaling ---
needed = ["GFTA_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.dropna(subset=needed).copy()

# --- Prepare weights from num_clips (normalize to mean=1, clip >0) ---
w = df_clean["num_clips"].astype(float).replace([np.inf, -np.inf], np.nan)
w = w.where(w > 0)
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()
df_clean["__w__"] = w
weights = df_clean["__w__"].values

# --- Scale predictors ---
scaler_age = StandardScaler()
scaler_med = StandardScaler()
scaler_cp  = StandardScaler()
scaler_gfta = StandardScaler()

df_clean["age_mos_scaled"] = scaler_age.fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = scaler_med.fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = scaler_cp.fit_transform(
    df_clean[["canonical_proportion"]]
)

# --- Scale outcome (GFTA) ---
df_clean["GFTA_Standard_scaled"] = scaler_gfta.fit_transform(
    df_clean[["GFTA_Standard"]]
)

# --- Final sanity: drop non-finite ---
predictors_base = ["age_mos_scaled", "gender_dummy", "Maternal_Education_Level_scaled"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

cols_needed_all = ["GFTA_Standard_scaled"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS weighted by num_clips) ---
gfta_baseline = fit_model("GFTA_Standard_scaled", predictors_base, df_clean, weights)
gfta_expanded = fit_model("GFTA_Standard_scaled", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = gfta_baseline.llf
ll_exp  = gfta_expanded.llf
k_base  = len(gfta_baseline.params)
k_exp   = len(gfta_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips → mean-normalized]: Demographics → GFTA (scaled)")
print(gfta_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips → mean-normalized]: Demographics + CP → GFTA (scaled)")
print(gfta_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion_scaled"
if term in gfta_expanded.params.index:
    cp_beta = gfta_expanded.params[term]
    cp_se   = gfta_expanded.bse[term]
    cp_t    = gfta_expanded.tvalues[term]
    cp_p    = gfta_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = gfta_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting GFTA *scaled*):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of GFTA (p < .05).\n"
          if cp_p < 0.05 else
          "❌ CP is not a statistically significant predictor of GFTA (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_p < 0.05 else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


### Weighted, Unscaled - [Weighted]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2
import os

# === File path (edit as needed) ===
master_csv = "tp1_tp1_master.csv"

# === Load data ===
if not os.path.exists(master_csv):
    raise FileNotFoundError(f"Could not find: {master_csv}")
df = pd.read_csv(master_csv)

# --- Helper to access columns '0' and '1' whether stored as int or str ---
def col(df, name_as_int_or_str):
    if name_as_int_or_str in df.columns:
        return df[name_as_int_or_str]
    # Flip type (int->str or str->int) if needed
    if isinstance(name_as_int_or_str, int) and str(name_as_int_or_str) in df.columns:
        return df[str(name_as_int_or_str)]
    if isinstance(name_as_int_or_str, str) and name_as_int_or_str.isdigit() and int(name_as_int_or_str) in df.columns:
        return df[int(name_as_int_or_str)]
    raise KeyError(f"Column '{name_as_int_or_str}' not found (checked as both int and str).")

# --- Compute num_clips = col 0 + col 1 ---
df["num_clips"] = pd.to_numeric(col(df, 0), errors="coerce") + pd.to_numeric(col(df, 1), errors="coerce")

# --- Coerce numeric columns (UNSCALED model uses raw values) ---
num_cols = ["GFTA_Standard", "canonical_proportion", "age_mos", "Maternal_Education_Level", "num_clips"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Keep rows with all needed fields (UNSCALED) ---
needed = ["GFTA_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=needed).copy()

# --- Weights from num_clips: normalize to mean = 1 (strictly > 0) ---
w = df_clean["num_clips"].astype(float)
w = w.where(w > 0)
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()
df_clean["__w__"] = w
weights = df_clean["__w__"].values

# --- Predictors (UNSCALED/raw) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion"]

# --- Final sanity: drop any residual non-finite values and align weights ---
cols_needed_all = ["GFTA_Standard"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
gfta_baseline = fit_model("GFTA_Standard", predictors_base, df_clean, weights)
gfta_expanded = fit_model("GFTA_Standard", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = gfta_baseline.llf
ll_exp  = gfta_expanded.llf
k_base  = len(gfta_baseline.params)
k_exp   = len(gfta_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics → GFTA_Standard")
print(gfta_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics + CP → GFTA_Standard")
print(gfta_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion"
if term in gfta_expanded.params.index:
    cp_beta = gfta_expanded.params[term]
    cp_se   = gfta_expanded.bse[term]
    cp_t    = gfta_expanded.tvalues[term]
    cp_p    = gfta_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = gfta_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting GFTA_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of GFTA (p < .05).\n"
          if cp_p < 0.05 else
          "❌ CP is not a statistically significant predictor of GFTA (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_p < 0.05 else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Sample size and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


## EVT

### Unweighted, Unscaled - [Expanded, Unscaled]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2

# === Load data ===
df = pd.read_csv("tp1_tp1_master.csv")  # adjust path to TP1 dataset

# --- Coerce numeric columns ---
num_cols = [
    "EVT_Standard", "canonical_proportion", "age_mos", "Maternal_Education_Level"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    # Try numeric 0/1
    try:
        v = float(x)
        if v in (0.0, 1.0):
            return int(v)  # 0 = Male, 1 = Female
    except Exception:
        pass
    # Strings
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields ---
needed = ["EVT_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy"]
df_clean = df.dropna(subset=needed).copy()

# --- Predictors (raw, unscaled) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion"]

# Ensure valid rows for both models
valid_rows = df_clean[
    ["EVT_Standard"] + predictors_exp
].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_rows].copy()

def fit_model(outcome_var, predictors, data):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    return sm.OLS(y.loc[mask], X.loc[mask]).fit()

# --- Fit models ---
evt_baseline = fit_model("EVT_Standard", predictors_base, df_clean)
evt_expanded = fit_model("EVT_Standard", predictors_exp, df_clean)

# --- Likelihood ratio test ---
ll_base = evt_baseline.llf
ll_exp  = evt_expanded.llf
k_base  = len(evt_baseline.params)
k_exp   = len(evt_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model (UNWEIGHTED, UNSCALED): Demographics → EVT_Standard")
print(evt_baseline.summary(), "\n")

print("Expanded Model (UNWEIGHTED, UNSCALED): Demographics + CP → EVT_Standard")
print(evt_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion"
if term in evt_expanded.params.index:
    cp_beta = evt_expanded.params[term]
    cp_se   = evt_expanded.bse[term]
    cp_t    = evt_expanded.tvalues[term]
    cp_p    = evt_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = evt_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting EVT_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of EVT (p < .05)."
          if cp_p < 0.05 else
          "❌ CP is not a statistically significant predictor of EVT (p ≥ .05).")
else:
    print("[Note] 'canonical_proportion' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_p < 0.05 else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Sample size ---
print(f"\nN used in both models: {df_clean.shape[0]}")


### Weighted, Scaled - [Expanded, Scaled]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2

# === Load CSV ===
df = pd.read_csv("tp1_tp1_master.csv")  # <-- replace with actual file path

# --- Compute num_clips by summing columns 0 and 1 (string-named) ---
df["num_clips"] = df[["0", "1"]].sum(axis=1)

# --- Coerce numeric columns ---
num_cols = [
    "EVT_Standard", "canonical_proportion", "age_mos",
    "Maternal_Education_Level", "num_clips"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Drop rows missing key fields BEFORE scaling ---
needed = ["EVT_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.dropna(subset=needed).copy()

# --- Prepare weights from num_clips (normalize to mean=1) ---
w = df_clean["num_clips"].astype(float).replace([np.inf, -np.inf], np.nan)
w = w.where(w > 0)
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()
df_clean["__w__"] = w

# --- Scale predictors ---
scaler_age = StandardScaler()
scaler_med = StandardScaler()
scaler_cp  = StandardScaler()

df_clean["age_mos_scaled"] = scaler_age.fit_transform(df_clean[["age_mos"]])
df_clean["Maternal_Education_Level_scaled"] = scaler_med.fit_transform(
    df_clean[["Maternal_Education_Level"]]
)
df_clean["canonical_proportion_scaled"] = scaler_cp.fit_transform(
    df_clean[["canonical_proportion"]]
)

# --- Final sanity: drop any non-finite values and align weights ---
predictors_base = ["age_mos_scaled", "gender_dummy", "Maternal_Education_Level_scaled"]
predictors_exp  = predictors_base + ["canonical_proportion_scaled"]

cols_needed_all = ["EVT_Standard"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS weighted by num_clips) ---
evt_baseline = fit_model("EVT_Standard", predictors_base, df_clean, weights)
evt_expanded = fit_model("EVT_Standard", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = evt_baseline.llf
ll_exp  = evt_expanded.llf
k_base  = len(evt_baseline.params)
k_exp   = len(evt_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips → mean-normalized]: Demographics → EVT_Standard")
print(evt_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips → mean-normalized]: Demographics + CP → EVT_Standard")
print(evt_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion_scaled"
if term in evt_expanded.params.index:
    cp_beta = evt_expanded.params[term]
    cp_se   = evt_expanded.bse[term]
    cp_t    = evt_expanded.tvalues[term]
    cp_p    = evt_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = evt_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting EVT_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of EVT (p < .05).\n"
          if cp_p < 0.05 else
          "❌ CP is not a statistically significant predictor of EVT (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion_scaled' not found in model parameters.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")
print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_p < 0.05 else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


### Weighted, Unscaled - [Weighted]

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import chi2
import os

# === File path (edit as needed) ===
master_csv = "tp1_tp1_master.csv"

# === Load data ===
if not os.path.exists(master_csv):
    raise FileNotFoundError(f"Could not find: {master_csv}")
df = pd.read_csv(master_csv)

# --- Helper to access columns '0' and '1' whether stored as int or str ---
def col(df, name_as_int_or_str):
    if name_as_int_or_str in df.columns:
        return df[name_as_int_or_str]
    if isinstance(name_as_int_or_str, int) and str(name_as_int_or_str) in df.columns:
        return df[str(name_as_int_or_str)]
    if isinstance(name_as_int_or_str, str) and name_as_int_or_str.isdigit() and int(name_as_int_or_str) in df.columns:
        return df[int(name_as_int_or_str)]
    raise KeyError(f"Column '{name_as_int_or_str}' not found (checked as both int and str).")

# --- Compute num_clips = col 0 + col 1 ---
df["num_clips"] = pd.to_numeric(col(df, 0), errors="coerce") + pd.to_numeric(col(df, 1), errors="coerce")

# --- Coerce numeric columns (UNSCALED model uses raw values) ---
num_cols = ["EVT_Standard", "canonical_proportion", "age_mos", "Maternal_Education_Level", "num_clips"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Robust gender dummy: handle 0/1 or strings ---
def to_gender_dummy(x):
    if pd.api.types.is_number(x):
        return int(x) if x in (0, 1) else np.nan  # 0 = Male, 1 = Female
    s = str(x).strip().lower()
    if s in ("f", "female"):
        return 1
    if s in ("m", "male"):
        return 0
    return np.nan

df["gender_dummy"] = df["gender"].apply(to_gender_dummy)

# --- Keep rows with all needed fields (UNSCALED) ---
needed = ["EVT_Standard", "canonical_proportion", "age_mos",
          "Maternal_Education_Level", "gender_dummy", "num_clips"]
df_clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=needed).copy()

# --- Weights from num_clips: normalize to mean = 1 (strictly > 0) ---
w = df_clean["num_clips"].astype(float)
w = w.where(w > 0)
if not w.notna().any():
    raise ValueError("No valid positive num_clips found for weighting.")
w = w / w.mean()
df_clean["__w__"] = w
weights = df_clean["__w__"].values

# --- Predictors (UNSCALED/raw) ---
predictors_base = ["age_mos", "gender_dummy", "Maternal_Education_Level"]
predictors_exp  = predictors_base + ["canonical_proportion"]

# --- Final sanity: drop any residual non-finite values and align weights ---
cols_needed_all = ["EVT_Standard"] + predictors_exp + ["__w__"]
valid_idx = df_clean[cols_needed_all].replace([np.inf, -np.inf], np.nan).dropna().index
df_clean = df_clean.loc[valid_idx].copy()
weights = df_clean["__w__"].values  # aligned to valid rows

def fit_model(outcome_var, predictors, data, weights):
    X = sm.add_constant(data[predictors])
    y = data[outcome_var]
    return sm.WLS(y, X, weights=weights).fit()

# --- Fit models (WLS by num_clips) ---
evt_baseline = fit_model("EVT_Standard", predictors_base, df_clean, weights)
evt_expanded = fit_model("EVT_Standard", predictors_exp,  df_clean, weights)

# --- Likelihood ratio test ---
ll_base = evt_baseline.llf
ll_exp  = evt_expanded.llf
k_base  = len(evt_baseline.params)
k_exp   = len(evt_expanded.params)

lrt_stat = 2 * (ll_exp - ll_base)
lrt_df   = k_exp - k_base
lrt_p    = chi2.sf(lrt_stat, lrt_df)

# --- Output summaries ---
print("Baseline Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics → EVT_Standard")
print(evt_baseline.summary(), "\n")

print("Expanded Model [WLS weighted by num_clips, UNSCALED predictors]: Demographics + CP → EVT_Standard")
print(evt_expanded.summary(), "\n")

# --- CP effect report ---
term = "canonical_proportion"
if term in evt_expanded.params.index:
    cp_beta = evt_expanded.params[term]
    cp_se   = evt_expanded.bse[term]
    cp_t    = evt_expanded.tvalues[term]
    cp_p    = evt_expanded.pvalues[term]
    cp_ci_l, cp_ci_u = evt_expanded.conf_int().loc[term].tolist()

    print("Canonical Proportion effect in Expanded Model (predicting EVT_Standard):")
    print(f"β = {cp_beta:.3f}  (SE = {cp_se:.3f}, t = {cp_t:.2f}, p = {cp_p:.4g})")
    print(f"95% CI: [{cp_ci_l:.3f}, {cp_ci_u:.3f}]")
    print("\nCP significance:",
          "✅ CP is a statistically significant predictor of EVT (p < .05).\n"
          if cp_p < 0.05 else
          "❌ CP is not a statistically significant predictor of EVT (p ≥ .05).\n")
else:
    print("[Note] 'canonical_proportion' not found in model parameters. Check predictors.\n")

# --- LRT report ---
print("Likelihood Ratio Test (Expanded vs Baseline):")
print(f"Log-Likelihood (Baseline): {ll_base:.4f}")
print(f"Log-Likelihood (Expanded): {ll_exp:.4f}")
print(f"LRT Statistic: {lrt_stat:.4f}")
print(f"Degrees of Freedom: {lrt_df}")
print(f"P-value: {lrt_p:.6f}")

print("\nModel comparison:",
      "✅ Expanded model significantly improves fit over Baseline (p < .05)."
      if lrt_p < 0.05 else
      "❌ Expanded model does not significantly improve fit over Baseline (p ≥ .05).")

# --- Sample size and weight summary ---
print(f"\nN used in both models: {df_clean.shape[0]}")
print(f"Weight summary (num_clips, normalized): mean={weights.mean():.3f}, "
      f"min={weights.min():.3f}, max={weights.max():.3f}")


## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Add num_clips = sum of columns "0" + "1"
df_clean["num_clips"] = df_clean[["0", "1"]].sum(axis=1)

# --- Build prediction grid ---
cp_grid = np.linspace(df_clean["canonical_proportion_scaled"].min(),
                      df_clean["canonical_proportion_scaled"].max(), 200)

Xp = pd.DataFrame({
    "const": 1.0,
    "age_mos": df_clean["age_mos"].mean(),
    "gender_dummy": 0,  # convention: male=0
    "Maternal_Education_Level": df_clean["Maternal_Education_Level"].mean(),
    "canonical_proportion": cp_grid
})
pred = gfta_expanded.get_prediction(Xp).summary_frame(alpha=0.05)

# --- Scatter points, size by num_clips ---
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(df_clean["canonical_proportion"],
           df_clean["GFTA_Standard"],
           s=np.clip(df_clean["num_clips"]/80, 10, 300),  # adjust scaling
           alpha=0.7, edgecolor="k", linewidth=0.4)

# --- Regression line + 95% CI ---
ax.plot(cp_grid, pred["mean"], color="blue", linewidth=2)
ax.fill_between(cp_grid, pred["mean_ci_lower"], pred["mean_ci_upper"],
                color="blue", alpha=0.2)

# --- Annotate β ---
beta = gfta_expanded.params["canonical_proportion_scaled"]
ax.text(0.05, 0.9, f"β = {beta:.2f}", transform=ax.transAxes,
        fontsize=12, fontweight="bold", color="blue")

# --- Labels ---
ax.set_xlabel("Canonical Proportion (Scaled)")
ax.set_ylabel("GFTA-2 (SS)")
ax.set_title("Canonical Proportion Predicting GFTA-2")

# --- Neaten style ---
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()
